# Quick run on Kaggle
Enable Internet and a GPU (T4 x2 works: two independent 16 GB workers). Add this codebase as a Kaggle dataset and add `sakhadib/bangla-bagdhara`. The quick run checks one encoder; it is not a full experiment. On CUDA the batched PyTorch probe engine is used automatically (`probe_backend: "auto"`).

In [ ]:
from pathlib import Path
import os, subprocess, sys
for variable in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[variable] = "2"
roots = sorted({p.parent for p in Path("/kaggle/input").rglob("run_research.py") if (p.parent / "idiom_subspace").is_dir()})
assert len(roots) == 1, f"Select the intended source directory explicitly: {roots}"
SRC = roots[0]
corpora = list(Path("/kaggle/input").rglob("*bagdhara*.json"))
assert len(corpora) == 1, f"Set DATA to the desired JSON: {corpora}"
DATA = corpora[0]
OUT = Path("/kaggle/working")
print(SRC, DATA)

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(SRC / "requirements.txt")])

In [ ]:
subprocess.check_call([sys.executable, str(SRC / "run_research.py"), "audit", "--data", str(DATA), "--out", str(OUT / "audit")])

In [ ]:
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
subprocess.check_call([sys.executable, str(SRC / "run_research.py"), "run", "--config", str(SRC / "configs/quick.json"),
                       "--data", str(DATA), "--out", str(OUT / "quick")], env=env)

## Optional: English literal glosses for S4a
`translate-literals` machine-translates the Bangla literal glosses (NLLB-200 by default) so that the literal-sense label L, and with it the figurative-literal separation analysis, becomes available. Rows are marked `approved=machine`; human-corrected rows (`approved=true`) take precedence.

In [ ]:
TRANSLATE = False
LITERALS = OUT / "literals" / "literal_glosses_mt.csv"
if TRANSLATE:
    subprocess.check_call([sys.executable, str(SRC / "run_research.py"), "translate-literals", "--config", str(SRC / "configs/full.json"),
                           "--data", str(DATA), "--out", str(LITERALS.parent)])

## Full run (optional)
One split seed and one taxonomy seed by default; replicate refits are opt-in (`--seeds 0,1,2 --label-seeds 0,1`, fresh output folder). This can exceed one Kaggle session: rerun the same command and output folder to resume (completed embeddings and readouts are reused). Each model job writes `job.log`; the launcher writes a live `launcher.json` queue.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    command = [sys.executable, str(SRC / "launch_gpu_jobs.py"), "--config", str(SRC / "configs/full.json"),
               "--data", str(DATA), "--out", str(OUT / "full"), "--gpus", "0,1"]
    if LITERALS.exists():
        command += ["--literal-reviews", str(LITERALS)]
    subprocess.check_call(command)

## Monitoring on Kaggle
A localhost dashboard is not reachable from your browser on Kaggle. Write a self-contained snapshot instead and download it from the output panel (it opens in any browser, no server). On a machine you can reach directly, use the live dashboard: `python dashboard.py --root OUTPUT --port 8787`.

In [ ]:
subprocess.check_call([sys.executable, str(SRC / "dashboard.py"), "--root", str(OUT), "--snapshot", str(OUT / "status.html")])

## Old embeddings, annotations and the decoder
Existing encoder caches can be reused: add `--cache-dir /kaggle/input/OLD_OUTPUT/cache` (and `--cache-only` to forbid loading weights). `templates` writes blank score/control files; it does not produce scores. `configs/decoder_3090.json` runs Qwen3-8B in bf16 and needs about 17 GB, so on a T4 add `"quantization": "4bit"` to its model entry (install `requirements-decoder.txt`). See `docs/RUNNING.md` and `docs/EXPERIMENTS.md`.